# Notebook 01: HotpotQA-Evaluationsdatensatz aufbauen

**Forschungsarbeit:** Evolutionäre Hyperparameter-Optimierung für RAG-Pipelines

Dieses Notebook deckt folgende Schritte ab:

1. Stratifizierte Stichprobe von 150 Fragen aus HotpotQA Distractor (Validation-Split) ziehen
2. Roh-Sample als JSON in `data/raw/` speichern
3. Korpus aus den Kontext-Paragraphen der Fragen aufbauen
4. Korpus als `.txt`-Dateien plus `corpus_meta.json` ablegen
5. QA-Paare mit aufgelösten supporting_facts speichern
6. Train/Test-Split 100/50 anlegen, stratifiziert nach Frage-Typ

Am Ende prüft eine Validierung Vollständigkeit und Konsistenz. Das Notebook läuft einmalig. Alle Artefakte landen unter `data/`.

## Setup

Liegt HotpotQA bereits im lokalen HuggingFace-Cache, schaltet das Setup auf Offline-Betrieb. Das überspringt den langsamen Online-Check beim Laden. Beim allerersten Lauf ohne Cache lädt das Notebook normal herunter.

In [1]:
import json
import os
import random
import re
from collections import Counter, defaultdict
from pathlib import Path

from src.environment import Environment

env = Environment()
env.make_dirs(env.raw_dir, env.processed_dir, env.corpus_dir)

PROJECT_ROOT = env.project_root
DATA_ROOT = env.data_root
RAW_DIR = env.raw_dir
PROCESSED_DIR = env.processed_dir
CORPUS_DIR = env.corpus_dir
QA_SPLIT_PATH = env.qa_split_path
CORPUS_META_PATH = env.corpus_meta_path

SEED = env.seed
N_TOTAL = 150
N_TRAIN = 100
N_TEST = 50

HF_DATASETS_CACHE = Path.home() / ".cache" / "huggingface" / "datasets"
if any(HF_DATASETS_CACHE.glob("hotpotqa___hotpot_qa*")):
    os.environ["HF_HUB_OFFLINE"] = "1"

print(env.summary())

experiment_name: ga_rag_optimizer_v1
seed: 42
raw_dir: data\raw
processed_dir: data\processed
corpus_dir: data\corpus
qa_split_path: data\processed\qa_pairs_split.json
corpus_meta_path: data\processed\corpus_meta.json
chroma_dir: data\processed\chroma_db
results_dir: results
logs_dir: results\logs
figures_dir: results\figures
tables_dir: results\tables


## Schritt 1: HotpotQA-Stichprobe ziehen

Der Validation-Split von HotpotQA Distractor wird geladen. Aus den 7405 Fragen werden 150 stratifiziert nach `type` (`bridge` vs. `comparison`) gezogen, damit das Originalverhältnis der beiden Frage-Typen erhalten bleibt.

In [2]:
from datasets import load_dataset

ds = load_dataset("hotpotqa/hotpot_qa", "distractor", split="validation")
print(f"Validation-Split gesamt: {len(ds)} Fragen")

by_type: dict[str, list[dict]] = defaultdict(list)
for ex in ds:
    row = dict(ex)
    by_type[row["type"]].append(row)

total = sum(len(v) for v in by_type.values())
rng = random.Random(SEED)

sample: list[dict] = []
for items in by_type.values():
    share = max(1, round(N_TOTAL * len(items) / total))
    rng.shuffle(items)
    sample.extend(items[:share])

rng.shuffle(sample)
sample = sample[:N_TOTAL]

print(f"\nGezogen: {len(sample)} Fragen")
print(f"Verteilung nach Typ: {dict(Counter(s['type'] for s in sample))}")

print("\nBeispiel-Frage:")
print(f"  Frage:   {sample[0]['question']}")
print(f"  Antwort: {sample[0]['answer']}")
print(f"  Typ:     {sample[0]['type']}")

Using the latest cached version of the dataset since hotpotqa/hotpot_qa couldn't be found on the Hugging Face Hub (offline mode is enabled).
Found the latest cached dataset configuration 'distractor' at C:\Users\MKari\.cache\huggingface\datasets\hotpotqa___hotpot_qa\distractor\0.0.0\1908d6afbbead072334abe2965f91bd2709910ab (last modified on Sat May 23 21:32:01 2026).


Validation-Split gesamt: 7405 Fragen

Gezogen: 150 Fragen
Verteilung nach Typ: {'bridge': 120, 'comparison': 30}

Beispiel-Frage:
  Frage:   At what theater is the composer and lyricist for the musical Big Fish a residential artist?
  Antwort: Ars Nova Theater
  Typ:     bridge


## Schritt 2: Roh-Sample speichern

Die 150 Fragen werden als JSON in `data/raw/` abgelegt. Damit ist die Stichprobe persistiert und reproduzierbar. Die Felder werden einzeln umkopiert, damit die Struktur unabhängig und stabil bleibt.

In [3]:
raw_file = RAW_DIR / "hotpotqa_validation_distractor_sample.json"

raw_data = [
    {
        "id": ex["id"],
        "question": ex["question"],
        "answer": ex["answer"],
        "type": ex["type"],
        "level": ex.get("level", ""),
        "supporting_facts": {
            "title": list(ex["supporting_facts"]["title"]),
            "sent_id": [int(i) for i in ex["supporting_facts"]["sent_id"]],
        },
        "context": {
            "title": list(ex["context"]["title"]),
            "sentences": [list(s) for s in ex["context"]["sentences"]],
        },
    }
    for ex in sample
]

raw_file.write_text(json.dumps(raw_data, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Gespeichert: {raw_file.relative_to(PROJECT_ROOT)}")
print(f"Größe: {raw_file.stat().st_size / 1024:.1f} KB")

Gespeichert: data\raw\hotpotqa_validation_distractor_sample.json
Größe: 1056.5 KB


## Schritt 3: Korpus aus den Kontext-Paragraphen extrahieren

Jede Frage bringt 10 Wikipedia-Paragraphen mit, 2 Belege und 8 Ablenker. Alle zusammen bilden den Korpus, wobei jeder Paragraph nur einmal aufgenommen wird. Aus dem Wikipedia-Titel entsteht die `paragraph_id`, ergeben zwei Titel denselben Namen, wird eine Nummer angehängt. Pro Paragraph wird markiert, ob er in mindestens einer Frage als Beleg dient (`is_supporting`).

In [4]:
corpus: dict[str, dict] = {}
title_to_pid: dict[str, str] = {}

for ex in sample:
    for title, sents in zip(ex["context"]["title"], ex["context"]["sentences"]):
        if title in title_to_pid:
            continue
        pid = re.sub(r"[^a-zA-Z0-9]+", "_", title).strip("_").lower() or "untitled"
        base, suffix = pid, 1
        while pid in corpus and corpus[pid]["title"] != title:
            suffix += 1
            pid = f"{base}_{suffix}"
        title_to_pid[title] = pid
        corpus[pid] = {
            "title": title,
            "sentences": list(sents),
            "n_sentences": len(sents),
        }

supporting_titles: set[str] = set()
for ex in sample:
    supporting_titles.update(ex["supporting_facts"]["title"])
for entry in corpus.values():
    entry["is_supporting"] = entry["title"] in supporting_titles

n_supporting = sum(1 for e in corpus.values() if e["is_supporting"])
print(f"Eindeutige Paragraphen im Korpus: {len(corpus)}")
print(f"Belegend (relevant für mindestens eine Frage): {n_supporting}")

example_pid = next(iter(corpus))
print(f"\nBeispiel-Paragraph '{example_pid}'")
print(f"  Titel:          {corpus[example_pid]['title']}")
print(f"  Sätze:          {corpus[example_pid]['n_sentences']}")
print(f"  Erster Satz:    {corpus[example_pid]['sentences'][0][:120]}")

Eindeutige Paragraphen im Korpus: 1498
Belegend (relevant für mindestens eine Frage): 300

Beispiel-Paragraph 'duet_all_night_long'
  Titel:          Duet All Night Long
  Sätze:          5
  Erster Satz:    Duet All Night Long is a split EP release by Reel Big Fish and Zolof the Rock & Roll Destroyer.


## Schritt 4: Korpus-Dateien schreiben

Pro Paragraph entsteht eine `.txt`-Datei unter `data/corpus/`. 

Format: Zeile 1 der Titel, dann eine Leerzeile, dann ein Satz pro Zeile. Zusätzlich wird `corpus_meta.json` mit allen Paragraph-Metadaten ohne Volltext geschrieben (für Statistik).

In [5]:
for pid, entry in corpus.items():
    file_path = CORPUS_DIR / f"{pid}.txt"
    body = "\n".join(entry["sentences"])
    file_path.write_text(f"{entry['title']}\n\n{body}", encoding="utf-8")

corpus_meta = {}
for pid, entry in corpus.items():
    corpus_meta[pid] = {
        "title": entry["title"],
        "n_sentences": entry["n_sentences"],
        "is_supporting": entry["is_supporting"],
    }

CORPUS_META_PATH.write_text(json.dumps(corpus_meta, ensure_ascii=False, indent=2), encoding="utf-8")

corpus_files = sorted(CORPUS_DIR.glob("*.txt"))
print(f"Korpus-Dateien geschrieben: {len(corpus_files)}")
print(f"Ort: {CORPUS_DIR.relative_to(PROJECT_ROOT)}")

example_file = corpus_files[0]
print(f"\nBeispiel-Datei {example_file.name}:")
print("-" * 60)
print(example_file.read_text(encoding="utf-8"))
print("-" * 60)

Korpus-Dateien geschrieben: 1498
Ort: data\corpus

Beispiel-Datei 0_4_0_0_4_0.txt:
------------------------------------------------------------
0-4-0+0-4-0

Under the Whyte notation for the classification of steam locomotives by wheel arrangement, the 0-4-0+0-4-0 is an articulated locomotive of the Garratt type.
 The wheel arrangement is effectively two 0-4-0 locomotives operating back-to-back or face-to-face, with the boiler and cab suspended between the two power units.
 Each power unit has no leading wheels, four powered and coupled driving wheels on two axles and no trailing wheels.
 A similar arrangement exists for Mallet and Meyer locomotives, but is referred to as 0-4-4-0.
------------------------------------------------------------


## Schritt 5: QA-Paare mit aufgelösten supporting_facts

HotpotQA sagt für jede Frage, in welchem Wikipedia-Paragraphen und in welchem Satz die Belege stehen. Hier wird der Titel jedes Belegs durch die `paragraph_id` aus Schritt 3 ersetzt. Damit tragen die Belege dieselben IDs wie die Korpus-Dateien, und die Metriken können beides direkt vergleichen.

In [6]:
qa_pairs: list[dict] = []
for ex in sample:
    titles = ex["supporting_facts"]["title"]
    sent_ids = ex["supporting_facts"]["sent_id"]

    supp = []
    for title, sent_id in zip(titles, sent_ids):
        if title not in title_to_pid:
            continue
        supp.append({"paragraph_id": title_to_pid[title], "sent_id": int(sent_id)})

    context_ids = []
    for title in ex["context"]["title"]:
        if title in title_to_pid:
            context_ids.append(title_to_pid[title])

    qa_pairs.append(
        {
            "id": ex["id"],
            "question": ex["question"],
            "answer": ex["answer"],
            "type": ex["type"],
            "level": ex.get("level", ""),
            "supporting_facts": supp,
            "context_paragraph_ids": context_ids,
        }
    )

qa_pairs_file = PROCESSED_DIR / "qa_pairs.json"
qa_pairs_file.write_text(json.dumps(qa_pairs, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"{len(qa_pairs)} QA-Paare geschrieben nach {qa_pairs_file.relative_to(PROJECT_ROOT)}")

print("\nBeispiel-QA-Paar:")
print(json.dumps(qa_pairs[0], indent=2, ensure_ascii=False))

150 QA-Paare geschrieben nach data\processed\qa_pairs.json

Beispiel-QA-Paar:
{
  "id": "5abdff775542993f32c2a082",
  "question": "At what theater is the composer and lyricist for the musical Big Fish a residential artist?",
  "answer": "Ars Nova Theater",
  "type": "bridge",
  "level": "hard",
  "supporting_facts": [
    {
      "paragraph_id": "big_fish_musical",
      "sent_id": 0
    },
    {
      "paragraph_id": "andrew_lippa",
      "sent_id": 1
    }
  ],
  "context_paragraph_ids": [
    "duet_all_night_long",
    "mystery_case_files_escape_from_ravenhearst",
    "mystery_case_files_dire_grove",
    "big_fish_musical",
    "burger_king_fish_sandwiches",
    "dan_regan",
    "aaron_barrett",
    "andrew_lippa",
    "hidden_expedition",
    "jeremy_lewis"
  ]
}


## Schritt 6: Train/Test-Split (stratifiziert)

100 Fragen bilden die Trainingsmenge für die GA-Fitness-Schleife, 50 die Testmenge für die Nachbewertung der besten Konfigurationen mit denselben Metriken. Keine Frage liegt in beiden Mengen. Beide Mengen behalten die Verteilung von `bridge` und `comparison`. Nach dem Split werden beide Mengen erneut gemischt, damit die Reihenfolge nicht nach Frage-Typ sortiert ist.

In [7]:
rng = random.Random(SEED)

by_type = defaultdict(list)
for p in qa_pairs:
    by_type[p["type"]].append(p)
total = len(qa_pairs)

train: list[dict] = []
test: list[dict] = []
for items in by_type.values():
    rng.shuffle(items)
    n_tr = round(N_TRAIN * len(items) / total)
    n_te = round(N_TEST * len(items) / total)
    train.extend(items[:n_tr])
    test.extend(items[n_tr : n_tr + n_te])

rng.shuffle(train)
rng.shuffle(test)
train = train[:N_TRAIN]
test = test[:N_TEST]

train_ids = {p["id"] for p in train}
test_ids = {p["id"] for p in test}
assert train_ids.isdisjoint(test_ids), "Trainings- und Testmenge überschneiden sich"

split = {"train": train, "test": test}
QA_SPLIT_PATH.write_text(json.dumps(split, ensure_ascii=False, indent=2), encoding="utf-8")

print(f"Split gespeichert: {QA_SPLIT_PATH.relative_to(PROJECT_ROOT)}")
print(f"  Training: {len(train)} Fragen, Verteilung {dict(Counter(p['type'] for p in train))}")
print(f"  Test:     {len(test)} Fragen, Verteilung {dict(Counter(p['type'] for p in test))}")

Split gespeichert: data\processed\qa_pairs_split.json
  Training: 100 Fragen, Verteilung {'bridge': 80, 'comparison': 20}
  Test:     50 Fragen, Verteilung {'bridge': 40, 'comparison': 10}


## Validierung

Abschließende Plausibilitäts-Prüfungen. Laufen alle durch, ist der Datensatz vollständig und konsistent.

In [8]:
n_corpus_files = sum(1 for _ in CORPUS_DIR.glob("*.txt"))
total_size_mb = sum(p.stat().st_size for p in DATA_ROOT.rglob("*") if p.is_file()) / 1024 / 1024

assert n_corpus_files >= len(corpus), "weniger Korpus-Dateien als Korpus-Einträge"
assert len(qa_pairs) == N_TOTAL, f"erwartet {N_TOTAL} QA-Paare, gefunden {len(qa_pairs)}"
assert len(train) == N_TRAIN, f"erwartet {N_TRAIN} Trainingsfragen, gefunden {len(train)}"
assert len(test) == N_TEST, f"erwartet {N_TEST} Testfragen, gefunden {len(test)}"

print("=== Zusammenfassung ===")
print(f"Korpus-Dateien:       {n_corpus_files}")
print(f"QA-Paare gesamt:      {len(qa_pairs)}")
print(f"Trainingsmenge:       {len(train)}")
print(f"Testmenge:            {len(test)}")
print(f"Größe data/ gesamt:   {total_size_mb:.2f} MB")
print("\nAlle Prüfungen bestanden.")

=== Zusammenfassung ===
Korpus-Dateien:       1498
QA-Paare gesamt:      150
Trainingsmenge:       100
Testmenge:            50
Größe data/ gesamt:   2.28 MB

Alle Prüfungen bestanden.
